# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [7]:
# --- The rule, in plain words (per skills/building-baselines) ---
# A content page is worth reviewing if it hasn't been touched in a long
# time (stale) but is STILL pulling in meaningful traffic (visible). That
# combination is real audience risk being ignored -- not a dead page
# nobody would miss, a live one quietly decaying. Ignoring it costs more
# than ignoring a stale page nobody sees.
#
# Reason code: "stale_but_visible" (named directly in the skill's own
# example -- using the session's vocabulary, not inventing a new one).
# Action label: "review_for_refresh". Staleness cutoff: 90 days, NOT
# the skill doc's illustrative 180 -- Signal 1 below shows why.
#
# Two signals this rule leans on, both tied to real FlyRank flags per the
# card:
#   1. staleness   -- behind the refresh flags
#   2. volume      -- behind the quick-win logic
# Both get their own bucket-table check with n printed, BEFORE they're
# allowed anywhere near the score -- a signal only earns a place in the
# rule if its bucket table backs it up.
#
# NOT going near trend_direction or trend_pct anywhere in this notebook:
# per skills/flyrank-data, is_declining_label is derived FROM
# trend_direction, which is computed FROM trend_pct. That's the same
# label-derived-column trap as gsc_clicks in ML-05 -- never a feature,
# never a scoring input, not even for the signal checks below.

import os

# Get the repo into this Colab session -- idempotent, safe to re-run.
REPO_URL = "https://github.com/joannadulay/FlyRank-ML-Internship.git"
REPO_DIR = "/content/FlyRank-ML-Internship"

os.chdir("/content")
if not os.path.isdir(REPO_DIR):
    !git clone {REPO_URL}
os.chdir(REPO_DIR)
print("Now in:", os.getcwd())

import pandas as pd
import glob

# Starter dataset per skills/flyrank-data -- one row per content item,
# trailing-90-day metrics, "ships in this repo." The fixed relative path
# failed on first run (FileNotFoundError) -- meaning either the repo
# isn't cloned into this session yet, or this notebook's working
# directory isn't the repo root. Searching instead of re-guessing a path.
FILENAME = "content_refresh_anonymized.csv"

print("Current working directory:", os.getcwd())
print("Top-level contents here:", os.listdir("."))

candidates = glob.glob(f"**/{FILENAME}", recursive=True)
# Also check the common Colab mount point in case the repo lives there.
candidates += glob.glob(f"/content/**/{FILENAME}", recursive=True)

if not candidates:
    raise FileNotFoundError(
        f"Could not find {FILENAME} anywhere under {os.getcwd()} or /content. "
        "The repo likely isn't cloned into this session yet -- in Colab, run "
        "`!git clone <your-repo-url>` in a cell first (or `%cd` into the repo "
        "if it's already cloned somewhere), then re-run this cell."
    )

CSV_PATH = candidates[0]
print("Found CSV at:", CSV_PATH)
if len(candidates) > 1:
    print("NOTE: found more than one match, using the first:", candidates)

df = pd.read_csv(CSV_PATH)

print("Shape:", df.shape)
print()
print("Columns:")
print(df.columns.tolist())
print()
print("Dtypes:")
print(df.dtypes)

# Real schema confirmed above (30000 x 44, matches the skill doc). The
# actual column is days_since_last_update, not the illustrative
# "days_since_update" from the skill's generic example. is_declining_label
# isn't in this raw file at all -- one less leakage risk to worry about.
# trend_direction / trend_pct ARE present but stay untouched, per the
# header comment above.

# ============================================================
# SIGNAL CHECK 1: staleness (behind the refresh flags)
# ============================================================
# Hypothesis: content that hasn't been updated in a long time is more
# likely to be losing traffic RIGHT NOW than fresh content. Checked using
# impressions_last_30d vs impressions_prev_30d -- both raw, same-
# timeframe columns. NOT trend_direction/trend_pct -- those stay off
# limits even here, diagnostics included.

staleness_bins = [-1, 89, 179, 364, float("inf")]
staleness_labels = ["<90d", "90-179d", "180-364d", "365d+"]
df["staleness_bucket"] = pd.cut(
    df["days_since_last_update"], bins=staleness_bins, labels=staleness_labels
)

df["recent_decline"] = df["impressions_last_30d"] < df["impressions_prev_30d"]

staleness_table = df.groupby("staleness_bucket", observed=True).agg(
    n=("content_id", "count"),
    pct_declining=("recent_decline", "mean"),
    mean_impr_last30=("impressions_last_30d", "mean"),
    mean_impr_prev30=("impressions_prev_30d", "mean"),
).round(3)

print("Signal 1 -- staleness vs recent decline (impressions_last_30d < impressions_prev_30d)")
print(staleness_table)
print()
# RESULTS: <90d n=20655 (61.5% declining), 90-179d n=9171 (75.6%),
# 180-364d n=169 (51.5%), 365d+ n=5 (60.0%).
#
# VERDICT: MIXED -- but the reason is the real finding. 29,826 of 30,000
# rows (99.4%) fall in the first two buckets; the 180-364d and 365d+
# buckets have only 169 and 5 rows, too few to trust their reversal.
# Within the two buckets that DO have real sample size, there's a clean,
# meaningful rise (61.5% -> 75.6% declining) right around the 90-day
# mark. The skill doc's illustrative ">=180 days" threshold would almost
# never fire on this dataset (0.6% of rows) -- the real, well-supported
# signal lives at >=90 days instead. Recalibrating the rule's staleness
# threshold to 90 days below, based on this measurement rather than the
# generic example.
print("VERDICT: MIXED (see comment above) -- staleness threshold recalibrated to 90 days")
print()

# ============================================================
# SIGNAL CHECK 2: volume (behind the quick-win logic)
# ============================================================
# Hypothesis: quick-win logic assumes there's a meaningful slice of
# HIGH-volume content still sitting in a POOR search position -- real
# traffic being left on the table by ranking, not by lack of demand.
# Checked using impressions_90d (volume) vs avg_position.
#
# avg_position == 0 is a documented sentinel for "no data" (per
# skills/flyrank-data, 1,205 rows in this dataset), NOT rank zero --
# excluded before computing any average. Same discipline as the
# gsc_avg_position == 0 flag from ML-05.

has_position_data = df["avg_position"] != 0
print(f"Rows with avg_position == 0 (no-data sentinel, excluded): {(~has_position_data).sum()}")

volume_df = df[has_position_data].copy()
volume_df["volume_bucket"] = pd.qcut(
    volume_df["impressions_90d"], q=4,
    labels=["Q1 (lowest)", "Q2", "Q3", "Q4 (highest)"]
)

volume_table = volume_df.groupby("volume_bucket", observed=True).agg(
    n=("content_id", "count"),
    mean_avg_position=("avg_position", "mean"),
    pct_poor_position=("avg_position", lambda s: (s > 10).mean()),
).round(3)

print()
print("Signal 2 -- volume vs search position (poor position defined as avg_position > 10)")
print(volume_table)
print()
# RESULTS: Q1 n=7202 (mean pos 16.4, 45.5% poor), Q2 n=7196 (21.6, 69.7%),
# Q3 n=7198 (16.6, 60.9%), Q4 n=7199 (13.5, 43.5%).
#
# VERDICT: CONFIRMED. Even Q4 (the highest-volume quartile) still has
# 43.5% of its pages sitting in a poor position (>10) -- high traffic
# does NOT mean the ranking problem is already solved. That's a real,
# sizable pool of high-visibility content with a fixable ranking issue --
# exactly the quick-win premise. (Q2's bump above Q1 and Q3 is odd and
# not fully explained here, but doesn't change the answer to the actual
# question being tested: does a meaningful quick-win pool exist at high
# volume? Yes.)
print("VERDICT: CONFIRMED (see comment above)")

# ============================================================
# What these two checks decided for the rule going into Section 2:
#   - Staleness threshold: 90 days, not the skill doc's illustrative
#     180 -- 180 barely occurs in this data (0.6% of rows), 90 is where
#     the real, well-supported signal actually lives.
#   - Volume: keeping this as a straightforward "visible" gate rather
#     than folding position into the score itself. Signal 2 confirms
#     WHY the rule is worth running (a real quick-win pool exists among
#     high-volume pages) without making position part of this baseline's
#     one simple rule -- keeping it to the single stale+visible
#     condition the plain-words rule already states.

Now in: /content/FlyRank-ML-Internship
Current working directory: /content/FlyRank-ML-Internship
Top-level contents here: ['skills', 'CLAUDE.md', 'assignments', '.gitignore', '.git', 'docs', 'GUIDE.md', 'LICENSE', 'work', 'requirements.txt', 'AGENTS.md', 'notebooks', 'data', 'scripts', 'submission', 'README.md', 'outputs', 'DATA_USE.md', 'SETUP.md', '.github']
Found CSV at: data/raw/content_refresh_anonymized.csv
NOTE: found more than one match, using the first: ['data/raw/content_refresh_anonymized.csv', '/content/FlyRank-ML-Internship/data/raw/content_refresh_anonymized.csv']
Shape: (30000, 44)

Columns:
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [12]:
# ============================================================
# THE RULE: score = is_stale AND is_visible, magnitude = impressions_90d
# ============================================================
# No fitted weights -- readable on purpose, per skills/building-baselines.
# Both gates come from Section 1's actual measurements, not the skill
# doc's illustrative numbers:
#   - stale threshold: 90 days (Signal 1's verdict -- 180 barely occurs
#     in this data)
#   - visible threshold: median impressions_90d, not the skill's
#     illustrative ">=500" -- same discipline, verified against this
#     dataset's real distribution instead of borrowed from the example.

STALE_DAYS_THRESHOLD = 90

visible_threshold = df["impressions_90d"].median()
print(f"impressions_90d median (visible threshold): {visible_threshold:.0f}")
print(df["impressions_90d"].describe())

df["is_stale"] = (df["days_since_last_update"] >= STALE_DAYS_THRESHOLD).astype(int)
df["is_visible"] = (df["impressions_90d"] >= visible_threshold).astype(int)

df["baseline_refresh_score"] = (
    df["is_stale"] * df["is_visible"] * df["impressions_90d"]
)

# ONE reason code, ONE action label -- applied only where the rule
# actually fires (score > 0). Everything else gets the "nothing to see
# here" pair, not a fabricated reason.
df["reason_code"] = "not_flagged"
df.loc[df["baseline_refresh_score"] > 0, "reason_code"] = "stale_but_visible"

df["suggested_action"] = "no_action"
df.loc[df["baseline_refresh_score"] > 0, "suggested_action"] = "review_for_refresh"

n_flagged = int((df["baseline_refresh_score"] > 0).sum())
print()
print(f"Flagged (stale AND visible): {n_flagged} of {len(df)} ({n_flagged / len(df) * 100:.1f}%)")

# ============================================================
# Rank and write the queue
# ============================================================
queue = df.sort_values("baseline_refresh_score", ascending=False).reset_index(drop=True)
queue["final_rank"] = queue.index + 1

import os

OUT_PATH = "work/outputs/baseline_action_score.csv"
os.makedirs(os.path.dirname(OUT_PATH), exist_ok=True)

output_cols = [
    "final_rank", "content_id", "client_id", "baseline_refresh_score",
    "reason_code", "suggested_action",
    "days_since_last_update", "impressions_90d", "is_stale", "is_visible",
]
queue[output_cols].to_csv(OUT_PATH, index=False)
print(f"Wrote {len(queue)} rows to {OUT_PATH}")

queue[output_cols].head(10)

impressions_90d median (visible threshold): 731
count     30000.000000
mean       5200.366300
std       16838.019547
min           1.000000
25%          81.000000
50%         731.000000
75%        3615.250000
max      517715.000000
Name: impressions_90d, dtype: float64

Flagged (stale AND visible): 5992 of 30000 (20.0%)
Wrote 30000 rows to work/outputs/baseline_action_score.csv


,final_rank,content_id,client_id,baseline_refresh_score,reason_code,suggested_action,days_since_last_update,impressions_90d,is_stale,is_visible
0,1,content_5fe46e04994d,client_4e07408562,517715,stale_but_visible,review_for_refresh,104,517715,1,1
1,2,content_2dba2b1f9536,client_6208ef0f77,443434,stale_but_visible,review_for_refresh,104,443434,1,1
2,3,content_2c2606c5d176,client_19581e27de,347399,stale_but_visible,review_for_refresh,104,347399,1,1
3,4,content_cb112fce36be,client_19581e27de,309910,stale_but_visible,review_for_refresh,104,309910,1,1
4,5,content_9532f197bbc8,client_4e07408562,309192,stale_but_visible,review_for_refresh,104,309192,1,1
5,6,content_36ff89c8214e,client_19581e27de,295097,stale_but_visible,review_for_refresh,104,295097,1,1
6,7,content_b28d1efd668f,client_6208ef0f77,286608,stale_but_visible,review_for_refresh,104,286608,1,1
7,8,content_813e88069237,client_6208ef0f77,233561,stale_but_visible,review_for_refresh,104,233561,1,1
8,9,content_c21024970297,client_19581e27de,211366,stale_but_visible,review_for_refresh,104,211366,1,1
9,10,content_c8e9d6ab9013,client_19581e27de,208678,stale_but_visible,review_for_refresh,104,208678,1,1


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [15]:
# ============================================================
# TOP-10 REVIEW
# ============================================================
# Pulling in avg_position and ctr for human-review context -- NOT
# trend_direction/trend_pct, which stay excluded from this notebook
# entirely (scoring AND review), per Section 1. avg_position == 0 is the
# documented "no data" sentinel, shown as such rather than as rank zero.

# Diagnostic: confirm avg_position is actually there before using it.
print("df has avg_position:", "avg_position" in df.columns)
print("df columns right now:", df.columns.tolist())
print()

# BUG FOUND ON FIRST RUN: top10 comes from queue, which inherits ALL of
# df's columns -- avg_position and ctr were never dropped. Merging them
# back in from df collided with the columns already there, and pandas
# silently renamed both sides to avg_position_x/avg_position_y instead
# of erroring -- the KeyError showed up one line later, when
# top10["avg_position"] no longer existed under that name. Fix: no merge
# needed, top10 already has what it needs.
top10 = queue.head(10).copy()

def position_note(pos):
    if pos == 0:
        return "No GSC position data -- can't tell if ranking is already fine; verify manually before refreshing."
    elif pos <= 5:
        return f"Already ranking well (avg position {pos:.1f}) despite being stale -- a refresh may not move the needle much."
    elif pos <= 10:
        return f"Ranking decently (avg position {pos:.1f}) -- traffic may hold up fine without a refresh."
    else:
        return f"Poor position (avg position {pos:.1f}) alongside high traffic -- but if the ranking problem is structural (competition, intent mismatch), a refresh alone won't fix it."

top10["why_here"] = top10.apply(
    lambda r: (f"Stale ({r['days_since_last_update']} days since update, >= {STALE_DAYS_THRESHOLD}d threshold) "
               f"and visible ({r['impressions_90d']:,} impressions_90d, above the "
               f"{visible_threshold:.0f} median) -- score is just impressions_90d since both gates are 1."),
    axis=1,
)
top10["what_would_make_it_wrong"] = top10["avg_position"].apply(position_note)

review_cols = [
    "final_rank", "content_id", "client_id", "baseline_refresh_score",
    "suggested_action", "why_here", "what_would_make_it_wrong",
]
for _, r in top10[review_cols].iterrows():
    print(f"#{r['final_rank']} -- {r['content_id']} ({r['client_id']})")
    print(f"  Action: {r['suggested_action']}")
    print(f"  Why here: {r['why_here']}")
    print(f"  What would make it wrong: {r['what_would_make_it_wrong']}")
    print()

# OBSERVATION carried over from Section 2's output: all 10 rows share
# days_since_last_update == 104, and only 3 distinct clients appear in
# the top 10. Checking whether that's a real cohort or a data artifact
# -- and it directly informs the "what would make it wrong" story above,
# not just Section 4's separate check.
n_at_104 = int((df["days_since_last_update"] == 104).sum())
n_clients_top10 = top10["client_id"].nunique()
print(f"Rows at exactly days_since_last_update == 104 (whole dataset): {n_at_104} of {len(df)}")
print(f"Distinct clients in the top 10: {n_clients_top10}")

df has avg_position: True
df columns right now: ['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct', 'staleness_bucket', 'recent_decline', 'is_stale', 'is_visible', 'baseline_refresh_score', 'reason_code', 'suggested_action']

#1 -- content_5fe46e04994d (client_4e07408562)
  Action

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [16]:
# ============================================================
# WEAK PICKS
# ============================================================

# --- Weak pick #1: the 104-day concentration, quantified ---
vc = df["days_since_last_update"].value_counts().sort_values(ascending=False)
print("Top 5 most common days_since_last_update values:")
print(vc.head(5))

n_at_104 = int((df["days_since_last_update"] == 104).sum())
n_bucket_90_179 = int(((df["days_since_last_update"] >= 90) & (df["days_since_last_update"] <= 179)).sum())
print()
print(f"n at exactly 104 days: {n_at_104} of {len(df)} ({n_at_104 / len(df) * 100:.1f}% of the WHOLE dataset)")
print(f"n at exactly 104 days as a share of the 90-179d bucket: "
      f"{n_at_104}/{n_bucket_90_179} = {n_at_104 / n_bucket_90_179 * 100:.1f}%")

print()
n_at_20 = int((df["days_since_last_update"] == 20).sum())
top5_sum = int(vc.head(5).sum())
print(f"n at exactly 20 days: {n_at_20} of {len(df)} -- LARGER than the 104 spike")
print(f"Top 5 values combined: {top5_sum}/{len(df)} = {top5_sum / len(df) * 100:.1f}% of the ENTIRE dataset")
print(f"Value 20 alone is {n_at_20}/20655 = {n_at_20 / 20655 * 100:.1f}% of the WHOLE under-90d bucket")
print()
print("FINDING (widened from the single-value version): this isn't just a 104-day")
print("artifact. 87.8% of the ENTIRE dataset sits on just 5 repeated")
print("days_since_last_update values (20, 104, 22, 8, 13). BOTH staleness buckets")
print("are each dominated by one spike, not organic update-timestamp variation.")
print("This behaves like a small set of batch/cohort stamps from how the synthetic")
print("dataset was generated, not a continuous real-world measurement.")
print()
print("Implication: STALE_DAYS_THRESHOLD=90 happens to cleanly separate the day-20")
print("batch (all below) from the day-104 batch (all above) -- so the stale gate is")
print("really discriminating BATCH vs BATCH, not a smooth staleness gradient.")
print("Signal 1's 61.5% vs 75.6% declining-rate difference is still a real,")
print("honestly-measured difference between these two specific cohorts -- but it")
print("may reflect whatever else differs between the batches (which clients, which")
print("content types got batched together) rather than staleness itself. Worth a")
print("follow-up check (e.g. does batch correlate with client_id or content_type)")
print("before the Week-5 model leans on this column as if it were continuous.")

# --- Weak pick #2: named rows that are probably false positives ---
print()
print("Named weak picks from the top 10 (rank, content_id, avg_position):")
print("  #1  content_5fe46e04994d  avg_position 4.2  -- already ranking well")
print("  #3  content_2c2606c5d176  avg_position 4.2  -- already ranking well")
print("  #5  content_9532f197bbc8  avg_position 2.0  -- already ranking well")
print("These three score highest purely because they have the most raw traffic")
print("(impressions_90d as the score's magnitude), but their position is already")
print("strong -- staleness by date does not necessarily mean anything is actually")
print("broken here. The rule has no way to see this; only the human review does.")

# --- Weak pick #3: outlier-driven ranking ---
print()
print(f"impressions_90d: mean {df['impressions_90d'].mean():.0f}, median "
      f"{df['impressions_90d'].median():.0f}, max {df['impressions_90d'].max():.0f}")
print("Because the score's magnitude is raw impressions_90d (heavily right-skewed,")
print("max is 700x the median), the top of the queue is dominated by a handful of")
print("outlier-scale clients rather than a diverse cross-section. Only 3 distinct")
print("clients appear across the entire top 10. Expected for a simple, transparent")
print("baseline -- but worth naming so the Week-5 model has something concrete to")
print("beat, not just a higher score.")

# ============================================================
# LEAKAGE CHECK
# ============================================================
# Explicit, checkable -- not just asserted in prose.

scoring_inputs = {"days_since_last_update", "impressions_90d"}
forbidden = {"trend_direction", "trend_pct", "is_declining_label"}

assert scoring_inputs.isdisjoint(forbidden), "LEAKAGE: forbidden column used in scoring"
assert "is_declining_label" not in df.columns, "unexpected: label column exists in this raw file"

print()
print("LEAKAGE CHECK")
print("Scoring inputs:", scoring_inputs)
print("Forbidden set (per skills/flyrank-data):", forbidden)
print("Disjoint -- confirmed, not just claimed.")
print()
print("No future-window inputs: both scoring columns are trailing/historical as of")
print("the snapshot (days_since_last_update, impressions_90d over the trailing 90")
print("days) -- nothing from impressions_last_30d/prev_30d or any forward-looking")
print("window was used in the score itself (those stayed inside Signal 1's")
print("diagnostic check only).")
print()
print("No product-flag/ID leakage: content_id and client_id appear in the output")
print("only as identifiers for the ranked queue -- never as scoring inputs.")

Top 5 most common days_since_last_update values:
days_since_last_update
20     11573
104     8773
22      3564
8       1929
13       515
Name: count, dtype: int64

n at exactly 104 days: 8773 of 30000 (29.2% of the WHOLE dataset)
n at exactly 104 days as a share of the 90-179d bucket: 8773/9171 = 95.7%

n at exactly 20 days: 11573 of 30000 -- LARGER than the 104 spike
Top 5 values combined: 26354/30000 = 87.8% of the ENTIRE dataset
Value 20 alone is 11573/20655 = 56.0% of the WHOLE under-90d bucket

FINDING (widened from the single-value version): this isn't just a 104-day
artifact. 87.8% of the ENTIRE dataset sits on just 5 repeated
days_since_last_update values (20, 104, 22, 8, 13). BOTH staleness buckets
are each dominated by one spike, not organic update-timestamp variation.
This behaves like a small set of batch/cohort stamps from how the synthetic
dataset was generated, not a continuous real-world measurement.

Implication: STALE_DAYS_THRESHOLD=90 happens to cleanly separate the 

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.